In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip -q install ultralytics opencv-python matplotlib

In [3]:
# ============================================================
# 12_pose_temporal_eval.ipynb
#
# Evaluación del filtro temporal de pose como etapa final
# del pipeline de detección de armas.
#
# Lógica:
#   - Solo se analiza pose cuando el detector de armas dispara
#   - Se aplica una ventana deslizante: si hay >= AIMING_RUN_THR
#     frames consecutivos (o en ventana) clasificados como AIMING
#     -> el clip se confirma como THREAT
#   - Objetivo: mantener TP de clips positivos y reducir FP
#     de clips negativos (especialmente N8/N9)
#
# BLOQUE A  — Clips POSITIVOS  (verificar que TP se mantienen)
# BLOQUE B  — Clips NEGATIVOS  (verificar que FP se reducen)
# BLOQUE C  — Métricas comparativas Modelo B vs Modelo B + HPE
# ============================================================

import os
import shutil
import math
import csv
from pathlib import Path
from collections import defaultdict, deque

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from ultralytics import YOLO

print('OK Imports')

OK Imports


In [4]:
# ============================================================
# CONFIG — ajusta aquí todos los parámetros
# ============================================================

POS_LIST = '/content/drive/MyDrive/TFM/datasets/videos/Gun_Action_Recognition_Dataset/splits/handgun_test.txt'
NEG_LIST = '/content/drive/MyDrive/TFM/datasets/videos/Gun_Action_Recognition_Dataset/splits/no_gun_test.txt'
OUT_DIR  = '/content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/pose_temporal_eval'

WEAPON_DET_WEIGHTS = '/content/drive/MyDrive/TFM/experiments/weapon_det/yolov8m_weapons_B_e50_640/weights/best.pt'
POSE_WEIGHTS       = 'yolov8m-pose.pt'

# Inferencia
IMG_SIZE    = 640
CONF_WEAPON = 0.25
IOU_NMS     = 0.7
CONF_POSE   = 0.5
FRAME_STRIDE = 1
MAX_SECONDS  = 15

# --- Parámetros del pipeline original (Modelo B) ---
# Mínimo de frames con arma para clasificar clip como positivo
DETECTION_THRESHOLD = 5

# --- Parámetros del filtro temporal de pose ---
KP_CONF_THR      = 0.3    # confianza mínima de keypoint
AIMING_ANGLE_THR = 160    # grados: brazo extendido -> AIMING

# Ventana deslizante (en frames con arma) para buscar AIMING sostenido
POSE_WINDOW      = 10     # analizar últimos N frames con arma detectada
# Mínimo de frames AIMING dentro de la ventana para confirmar THREAT
AIMING_RUN_THR   = 3      # prueba también con 2, 4, 5

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print('OK Config')
print(f'  POSE_WINDOW={POSE_WINDOW}  AIMING_RUN_THR={AIMING_RUN_THR}')

OK Config
  POSE_WINDOW=10  AIMING_RUN_THR=3


In [5]:
# ============================================================
# KEYPOINTS COCO + HELPERS GEOMÉTRICOS
# ============================================================

L_SHOULDER, R_SHOULDER = 5, 6
L_ELBOW,    R_ELBOW    = 7, 8
L_WRIST,    R_WRIST    = 9, 10

def angle_between(a, b, c):
    ba = (a[0]-b[0], a[1]-b[1])
    bc = (c[0]-b[0], c[1]-b[1])
    dot  = ba[0]*bc[0] + ba[1]*bc[1]
    norm = math.sqrt(ba[0]**2+ba[1]**2) * math.sqrt(bc[0]**2+bc[1]**2)
    if norm < 1e-6:
        return None
    return math.degrees(math.acos(max(-1.0, min(1.0, dot/norm))))

def get_arm_angles(kps, confs):
    results = {'left': None, 'right': None}
    for side, (sh, el, wr) in [('left',  (L_SHOULDER, L_ELBOW, L_WRIST)),
                                 ('right', (R_SHOULDER, R_ELBOW, R_WRIST))]:
        if all(confs[i] >= KP_CONF_THR for i in [sh, el, wr]):
            results[side] = angle_between(tuple(kps[sh]), tuple(kps[el]), tuple(kps[wr]))
    return results

def classify_pose(arm_angles):
    valid = {k: v for k, v in arm_angles.items() if v is not None}
    if not valid:
        return 'NEUTRAL'
    if any(v >= AIMING_ANGLE_THR for v in valid.values()):
        return 'AIMING'
    return 'HOLDING'

def bbox_center(box):
    return ((box[0]+box[2])/2, (box[1]+box[3])/2)

def find_nearest_person(weapon_box, person_boxes):
    if not person_boxes:
        return None
    wc = bbox_center(weapon_box)
    return int(np.argmin([math.dist(wc, bbox_center(pb)) for pb in person_boxes]))

print('OK Helpers geométricos')

OK Helpers geométricos


In [6]:
weapon_model = YOLO(WEAPON_DET_WEIGHTS)
pose_model   = YOLO(POSE_WEIGHTS)
print('OK Modelos cargados')

OK Modelos cargados


In [7]:
# ============================================================
# FUNCIÓN PRINCIPAL: procesar un clip con pipeline completo
# Devuelve métricas del Modelo B solo y del pipeline B+HPE
# ============================================================

def process_clip(video_path):
    """
    Procesa un clip frame a frame.
    Retorna un dict con:
      - frames_with_weapon : int  (para Modelo B original)
      - pred_modelB        : 0/1  (clasificación Modelo B)
      - threat_confirmed   : bool (True si el filtro temporal de pose confirma THREAT)
      - frame_log          : lista de dicts por frame (para análisis)
    """
    local_in = '/content/tmp_eval.mp4'
    shutil.copy2(str(video_path), local_in)

    cap      = cv2.VideoCapture(local_in)
    fps      = cap.get(cv2.CAP_PROP_FPS) or 30
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    max_f    = min(n_frames, int(MAX_SECONDS * fps))

    frames_with_weapon = 0
    threat_confirmed   = False
    frame_log          = []

    # Ventana deslizante: guarda los últimos POSE_WINDOW labels de pose
    # (solo cuando hay arma en el frame)
    pose_window = deque(maxlen=POSE_WINDOW)

    frame_i = 0
    while frame_i < max_f:
        ok, frame = cap.read()
        if not ok:
            break
        if frame_i % FRAME_STRIDE != 0:
            frame_i += 1
            continue

        # --- Detección de armas ---
        wr = weapon_model.predict(frame, imgsz=IMG_SIZE, conf=CONF_WEAPON,
                                   iou=IOU_NMS, verbose=False, device='cuda')[0]
        weapon_boxes = []
        if wr.boxes is not None and len(wr.boxes) > 0:
            for b in wr.boxes:
                x1,y1,x2,y2 = map(float, b.xyxy[0])
                weapon_boxes.append((x1,y1,x2,y2,float(b.conf[0])))

        if weapon_boxes:
            frames_with_weapon += 1

            # --- Estimación de pose (solo si hay arma) ---
            pr = pose_model.predict(frame, imgsz=IMG_SIZE, conf=CONF_POSE,
                                     verbose=False, device='cuda')[0]

            pose_label = 'NEUTRAL'
            person_boxes_raw = []

            if pr.keypoints is not None and len(pr.keypoints) > 0:
                kps_data  = pr.keypoints.xy.cpu().numpy()
                conf_data = pr.keypoints.conf.cpu().numpy()
                if pr.boxes is not None:
                    for b in pr.boxes:
                        person_boxes_raw.append(list(map(float, b.xyxy[0])))

                # Asociar arma a persona más cercana
                nearest = find_nearest_person(weapon_boxes[0][:4], person_boxes_raw)
                if nearest is not None and nearest < len(kps_data):
                    angles    = get_arm_angles(kps_data[nearest], conf_data[nearest])
                    pose_label = classify_pose(angles)

            pose_window.append(pose_label)

            # Comprobar si la ventana actual tiene suficientes AIMING
            n_aiming_in_window = sum(1 for p in pose_window if p == 'AIMING')
            if n_aiming_in_window >= AIMING_RUN_THR:
                threat_confirmed = True

            frame_log.append({
                'frame':            frame_i,
                'has_weapon':       True,
                'pose_label':       pose_label,
                'aiming_in_window': n_aiming_in_window,
                'threat_so_far':    threat_confirmed,
            })

        frame_i += 1

    cap.release()
    os.remove(local_in)

    pred_modelB = 1 if frames_with_weapon >= DETECTION_THRESHOLD else 0

    return {
        'frames_with_weapon': frames_with_weapon,
        'pred_modelB':        pred_modelB,
        'threat_confirmed':   threat_confirmed,
        'frame_log':          frame_log,
    }

print('OK Función process_clip definida')

OK Función process_clip definida


In [8]:
# ============================================================
# BLOQUE A — Clips POSITIVOS
# Verificar que el filtro de pose no elimina TPs del Modelo B
# ============================================================
print('=' * 60)
print('BLOQUE A — Clips positivos')
print('=' * 60)

pos_paths = [Path(l.strip()) for l in Path(POS_LIST).read_text().splitlines() if l.strip()]

pos_results = []
for vp in pos_paths:
    if not vp.exists():
        continue
    clip_id = vp.parent.name
    r = process_clip(vp)
    r['clip_id'] = clip_id
    r['true']    = 1
    pos_results.append(r)

    # Etiquetas para impresión
    b_label  = 'TP' if r['pred_modelB'] == 1 else 'FN'
    hp_label = 'TP' if r['threat_confirmed'] else 'FN'
    print(f"  {clip_id:<35}  ModeloB={b_label}  B+HPE={hp_label}  "
          f"(arma={r['frames_with_weapon']}f, threat={'YES' if r['threat_confirmed'] else 'NO'})")

# Resumen
tp_b   = sum(1 for r in pos_results if r['pred_modelB'] == 1)
fn_b   = sum(1 for r in pos_results if r['pred_modelB'] == 0)
tp_hpe = sum(1 for r in pos_results if r['threat_confirmed'])
fn_hpe = sum(1 for r in pos_results if not r['threat_confirmed'])

print(f'\n  --- Resumen positivos ({len(pos_results)} clips) ---')
print(f'  Modelo B solo :  TP={tp_b}  FN={fn_b}')
print(f'  B + HPE       :  TP={tp_hpe}  FN={fn_hpe}')
print(f'  Recall ModeloB  : {tp_b/(tp_b+fn_b):.4f}')
print(f'  Recall B+HPE    : {tp_hpe/(tp_hpe+fn_hpe):.4f}')

BLOQUE A — Clips positivos
  PAH1_C1_P1_V1_HB_3                   ModeloB=TP  B+HPE=TP  (arma=25f, threat=YES)
  PAH1_C1_P1_V1_HB_4                   ModeloB=TP  B+HPE=TP  (arma=80f, threat=YES)
  PAH1_C1_P2_V1_HB_1                   ModeloB=TP  B+HPE=TP  (arma=144f, threat=YES)
  PAH1_C1_P2_V1_HB_3                   ModeloB=TP  B+HPE=TP  (arma=133f, threat=YES)
  PAH1_C1_P4_V1_HB_1                   ModeloB=TP  B+HPE=TP  (arma=64f, threat=YES)
  PAH1_C1_P4_V1_HB_2                   ModeloB=TP  B+HPE=TP  (arma=64f, threat=YES)
  PAH1_C2_P3_V1_HB_1                   ModeloB=TP  B+HPE=TP  (arma=42f, threat=YES)
  PAH1_C2_P3_V1_HB_3                   ModeloB=TP  B+HPE=TP  (arma=30f, threat=YES)
  PAH1_C2_P3_V2_HB_2                   ModeloB=TP  B+HPE=TP  (arma=45f, threat=YES)
  PAH1_C2_P3_V2_HB_3                   ModeloB=TP  B+HPE=TP  (arma=87f, threat=YES)
  PAH1_C2_P5_V1_HB_1                   ModeloB=TP  B+HPE=TP  (arma=70f, threat=YES)
  PAH1_C2_P5_V1_HB_4                   ModeloB=

In [9]:
# ============================================================
# BLOQUE B — Clips NEGATIVOS
# Verificar cuántos FP del Modelo B elimina el filtro de pose
# ============================================================
print('=' * 60)
print('BLOQUE B — Clips negativos')
print('=' * 60)

neg_paths = [Path(l.strip()) for l in Path(NEG_LIST).read_text().splitlines() if l.strip()]

neg_results = []
for vp in neg_paths:
    if not vp.exists():
        continue
    clip_id  = vp.parent.name
    category = clip_id.split('_')[0]
    r = process_clip(vp)
    r['clip_id']  = clip_id
    r['category'] = category
    r['true']     = 0
    neg_results.append(r)

    # Solo mostrar clips donde el Modelo B ya disparó (FP potenciales)
    if r['pred_modelB'] == 1:
        resolved = 'TN (filtrado)' if not r['threat_confirmed'] else 'FP (persiste)'
        print(f"  {clip_id:<35}  ModeloB=FP  B+HPE={resolved}  "
              f"(arma={r['frames_with_weapon']}f)")

# Resumen global negativos
fp_b   = sum(1 for r in neg_results if r['pred_modelB'] == 1)
tn_b   = sum(1 for r in neg_results if r['pred_modelB'] == 0)
# De los FP del modelo B, cuántos resuelve HPE
fp_resolved  = sum(1 for r in neg_results if r['pred_modelB'] == 1 and not r['threat_confirmed'])
fp_remaining = sum(1 for r in neg_results if r['pred_modelB'] == 1 and r['threat_confirmed'])
# TN del modelo B que HPE convierte en FP (falsos negativos de HPE sobre negativos)
new_fp = sum(1 for r in neg_results if r['pred_modelB'] == 0 and r['threat_confirmed'])

print(f'\n  --- Resumen negativos ({len(neg_results)} clips) ---')
print(f'  Modelo B: FP={fp_b}  TN={tn_b}')
print(f'  De los {fp_b} FP del ModeloB:')
print(f'    Resueltos por HPE (pasan a TN) : {fp_resolved}  ({fp_resolved/fp_b*100:.1f}%)')
print(f'    Persisten como FP              : {fp_remaining}  ({fp_remaining/fp_b*100:.1f}%)')
if new_fp > 0:
    print(f'  Nuevos FP introducidos por HPE  : {new_fp}  (TN->FP)')

BLOQUE B — Clips negativos
  N10_C1_P4_V1_HB_2                    ModeloB=FP  B+HPE=FP (persiste)  (arma=25f)
  N10_C2_P3_V2_HB_2                    ModeloB=FP  B+HPE=FP (persiste)  (arma=8f)
  N11_C1_P2_V1_HB_2                    ModeloB=FP  B+HPE=FP (persiste)  (arma=8f)
  N11_C1_P4_V1_HB_2                    ModeloB=FP  B+HPE=FP (persiste)  (arma=22f)
  N11_C2_P3_V2_HB_2                    ModeloB=FP  B+HPE=FP (persiste)  (arma=12f)
  N11_C2_P5_V3_HB_1                    ModeloB=FP  B+HPE=FP (persiste)  (arma=12f)
  N12_C1_P4_V1_HB_1                    ModeloB=FP  B+HPE=FP (persiste)  (arma=27f)
  N12_C2_P5_V1_HB_1                    ModeloB=FP  B+HPE=FP (persiste)  (arma=10f)
  N12_C2_P5_V2_HB_3                    ModeloB=FP  B+HPE=TN (filtrado)  (arma=11f)
  N12_C2_P5_V3_HB_1                    ModeloB=FP  B+HPE=TN (filtrado)  (arma=12f)
  N1_C1_P1_V1_HB_1                     ModeloB=FP  B+HPE=FP (persiste)  (arma=12f)
  N1_C1_P4_V1_HB_1                     ModeloB=FP  B+HPE=FP (p

In [10]:
# ============================================================
# BLOQUE C — Métricas comparativas y análisis por categoría
# ============================================================
print('=' * 60)
print('BLOQUE C — Métricas comparativas')
print('=' * 60)

all_results = pos_results + neg_results

# --- Modelo B original ---
TP_b = sum(1 for r in all_results if r['true']==1 and r['pred_modelB']==1)
TN_b = sum(1 for r in all_results if r['true']==0 and r['pred_modelB']==0)
FP_b = sum(1 for r in all_results if r['true']==0 and r['pred_modelB']==1)
FN_b = sum(1 for r in all_results if r['true']==1 and r['pred_modelB']==0)

acc_b  = (TP_b+TN_b) / len(all_results)
prec_b = TP_b / (TP_b+FP_b) if (TP_b+FP_b) > 0 else 0
rec_b  = TP_b / (TP_b+FN_b) if (TP_b+FN_b) > 0 else 0
f1_b   = 2*prec_b*rec_b / (prec_b+rec_b) if (prec_b+rec_b) > 0 else 0

# --- Modelo B + HPE ---
# Un clip se predice positivo si: ModeloB=1 AND threat_confirmed=True
def pred_hpe(r):
    return 1 if (r['pred_modelB'] == 1 and r['threat_confirmed']) else 0

TP_h = sum(1 for r in all_results if r['true']==1 and pred_hpe(r)==1)
TN_h = sum(1 for r in all_results if r['true']==0 and pred_hpe(r)==0)
FP_h = sum(1 for r in all_results if r['true']==0 and pred_hpe(r)==1)
FN_h = sum(1 for r in all_results if r['true']==1 and pred_hpe(r)==0)

acc_h  = (TP_h+TN_h) / len(all_results)
prec_h = TP_h / (TP_h+FP_h) if (TP_h+FP_h) > 0 else 0
rec_h  = TP_h / (TP_h+FN_h) if (TP_h+FN_h) > 0 else 0
f1_h   = 2*prec_h*rec_h / (prec_h+rec_h) if (prec_h+rec_h) > 0 else 0

print(f'\n  {"":20} {"Modelo B":>10} {"B + HPE":>10}  Delta')
print('  ' + '-'*52)
print(f'  {"Accuracy":20} {acc_b:>10.4f} {acc_h:>10.4f}  {acc_h-acc_b:+.4f}')
print(f'  {"Precision":20} {prec_b:>10.4f} {prec_h:>10.4f}  {prec_h-prec_b:+.4f}')
print(f'  {"Recall":20} {rec_b:>10.4f} {rec_h:>10.4f}  {rec_h-rec_b:+.4f}')
print(f'  {"F1":20} {f1_b:>10.4f} {f1_h:>10.4f}  {f1_h-f1_b:+.4f}')
print(f'  {"TP":20} {TP_b:>10} {TP_h:>10}  {TP_h-TP_b:+}')
print(f'  {"TN":20} {TN_b:>10} {TN_h:>10}  {TN_h-TN_b:+}')
print(f'  {"FP":20} {FP_b:>10} {FP_h:>10}  {FP_h-FP_b:+}')
print(f'  {"FN":20} {FN_b:>10} {FN_h:>10}  {FN_h-FN_b:+}')

# --- FP por categoría negativa ---
print(f'\n  --- FP por categoría negativa ---')
cat_labels = {
    'N1':'Walking empty hands',  'N2':'Jogging',
    'N3':'Running',              'N4':'Sneaking empty hands',
    'N5':'Phone relaxed',        'N6':'Phone looking',
    'N7':'Phone both hands',     'N8':'Phone recording 1h',
    'N9':'Phone recording 2h',   'N10':'Water bottle relaxed',
    'N11':'Drinking',            'N12':'Holding heavy object',
}
cat_stats = defaultdict(lambda: {'total':0,'fp_b':0,'fp_hpe':0})
for r in neg_results:
    cat = r['category']
    cat_stats[cat]['total'] += 1
    if r['pred_modelB'] == 1:
        cat_stats[cat]['fp_b'] += 1
    if pred_hpe(r) == 1:
        cat_stats[cat]['fp_hpe'] += 1

print(f"  {'Cat':<5} {'Descripción':<24} {'Tot':>4}  "
      f"{'FP_B':>6} {'%B':>6}  {'FP_HPE':>6} {'%HPE':>6}  {'Mejora':>7}")
print('  ' + '-'*72)
for cat in sorted(cat_stats, key=lambda x: int(x[1:])):
    s = cat_stats[cat]
    tot = s['total']
    pct_b   = s['fp_b']   / tot * 100 if tot > 0 else 0
    pct_hpe = s['fp_hpe'] / tot * 100 if tot > 0 else 0
    mejora  = pct_b - pct_hpe
    desc    = cat_labels.get(cat, '')
    print(f"  {cat:<5} {desc:<24} {tot:>4}  "
          f"{s['fp_b']:>6} {pct_b:>6.1f}%  {s['fp_hpe']:>6} {pct_hpe:>6.1f}%  {mejora:>+6.1f}pp")

BLOQUE C — Métricas comparativas

                         Modelo B    B + HPE  Delta
  ----------------------------------------------------
  Accuracy                 0.7519     0.7403  -0.0116
  Precision                0.7209     0.7829  +0.0620
  Recall                   0.8857     0.7214  -0.1643
  F1                       0.7949     0.7509  -0.0439
  TP                          124        101  -23
  TN                           70         90  +20
  FP                           48         28  -20
  FN                           16         39  +23

  --- FP por categoría negativa ---
  Cat   Descripción               Tot    FP_B     %B  FP_HPE   %HPE   Mejora
  ------------------------------------------------------------------------
  N1    Walking empty hands         9       3   33.3%       3   33.3%    +0.0pp
  N2    Jogging                     9       3   33.3%       0    0.0%   +33.3pp
  N3    Running                     8       0    0.0%       0    0.0%    +0.0pp
  N4    Sneaki

In [11]:
# ============================================================
# BLOQUE D — Guardar resultados en Drive
# ============================================================

results_path = Path(OUT_DIR) / 'pose_temporal_results.txt'
with open(results_path, 'w') as f:
    f.write('=== EVALUACIÓN B + HPE TEMPORAL ===\n\n')
    f.write(f'POSE_WINDOW={POSE_WINDOW}  AIMING_RUN_THR={AIMING_RUN_THR}\n')
    f.write(f'AIMING_ANGLE_THR={AIMING_ANGLE_THR}  KP_CONF_THR={KP_CONF_THR}\n\n')
    f.write(f'  Accuracy   ModeloB={acc_b:.4f}  B+HPE={acc_h:.4f}  ({acc_h-acc_b:+.4f})\n')
    f.write(f'  Precision  ModeloB={prec_b:.4f}  B+HPE={prec_h:.4f}  ({prec_h-prec_b:+.4f})\n')
    f.write(f'  Recall     ModeloB={rec_b:.4f}  B+HPE={rec_h:.4f}  ({rec_h-rec_b:+.4f})\n')
    f.write(f'  F1         ModeloB={f1_b:.4f}  B+HPE={f1_h:.4f}  ({f1_h-f1_b:+.4f})\n')
    f.write(f'  TP={TP_h} TN={TN_h} FP={FP_h} FN={FN_h}\n\n')
    f.write('--- FP por categoria ---\n')
    for cat in sorted(cat_stats, key=lambda x: int(x[1:])):
        s = cat_stats[cat]
        tot = s['total']
        pct_b   = s['fp_b']   / tot * 100 if tot > 0 else 0
        pct_hpe = s['fp_hpe'] / tot * 100 if tot > 0 else 0
        f.write(f'  {cat}: FP_B={s["fp_b"]}/{tot} ({pct_b:.1f}%)  FP_HPE={s["fp_hpe"]}/{tot} ({pct_hpe:.1f}%)\n')

# CSV detallado por clip
csv_path = Path(OUT_DIR) / 'clip_results.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=[
        'clip_id','true','category','frames_with_weapon',
        'pred_modelB','threat_confirmed','pred_hpe'
    ])
    writer.writeheader()
    for r in all_results:
        writer.writerow({
            'clip_id':           r['clip_id'],
            'true':              r['true'],
            'category':          r.get('category','POS'),
            'frames_with_weapon':r['frames_with_weapon'],
            'pred_modelB':       r['pred_modelB'],
            'threat_confirmed':  int(r['threat_confirmed']),
            'pred_hpe':          pred_hpe(r),
        })

print(f'Resultados guardados en: {results_path}')
print(f'CSV guardado en        : {csv_path}')
print('Evaluacion completa.')

Resultados guardados en: /content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/pose_temporal_eval/pose_temporal_results.txt
CSV guardado en        : /content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/pose_temporal_eval/clip_results.csv
Evaluacion completa.
